# Agent Memory Deep Dive: Checkpointers vs. Long-Term Memory

This notebook is a companion to `notebook.ipynb` and `langgraph_advanced.ipynb`
in this same folder. It goes deep on one topic those notebooks only touch:
**how an agent remembers things, at two different timescales.**

By the end you will have built, with real code against a real Claude API:

1. **Short-term (thread-scoped) memory** via LangGraph checkpointers —
   `InMemorySaver` and `SqliteSaver`, compared side by side, including
   surviving an `interrupt()` and a real process restart.
2. **The amnesia problem** — why a new conversation thread means the agent
   remembers nothing, no matter how good the checkpointer is.
3. **Long-term (cross-thread) memory** via Mem0 + ChromaDB — a "profile
   memory" layer that persists facts about a user across every future
   conversation, not just the current one.
4. **Alternatives to Mem0** and when you'd reach for something else instead.
5. **A combined finale** showing both memory systems working together in
   one realistic customer-support scenario.

## Prerequisites

This notebook assumes you've already seen step (d) "Checkpointing / memory"
in `notebook.ipynb` — the *what* of checkpointing. This notebook is the
*how it actually works under the hood, compared across implementations,
and what to layer on top of it*.

## Setup


In [ ]:
import os
import time
import sqlite3
import warnings
import logging
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
logging.getLogger("chromadb").setLevel(logging.ERROR)
logging.getLogger("posthog").setLevel(logging.ERROR)
logging.getLogger("mem0").setLevel(logging.ERROR)

# Disable Mem0's anonymous telemetry before importing anything mem0-related —
# otherwise every call blocks briefly on a PostHog ping that often times out
# on restricted networks and clutters output with noisy tracebacks.
os.environ["MEM0_TELEMETRY"] = "False"

# Load the repo-root .env (same convention as notebook.ipynb / langgraph_advanced.ipynb)
load_dotenv("../../.env")

# Single visible flag controlling which provider the whole notebook uses —
# same convention as the other two notebooks in this folder. No silent
# auto-detection: the matching key must be present in .env.
PROVIDER = "anthropic"  # or "openai"

from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI


def get_llm():
    # Note: claude-sonnet-5 has deprecated the temperature param (returns a 400
    # if passed) — omit it for Anthropic. OpenAI still accepts it.
    if PROVIDER == "anthropic":
        key = os.getenv("ANTHROPIC_API_KEY")
        if not key:
            raise RuntimeError("ANTHROPIC_API_KEY not set in .env — cannot proceed (no mock mode).")
        return ChatAnthropic(model="claude-sonnet-5", api_key=key)
    elif PROVIDER == "openai":
        key = os.getenv("OPENAI_API_KEY")
        if not key:
            raise RuntimeError("OPENAI_API_KEY not set in .env — cannot proceed (no mock mode).")
        return ChatOpenAI(model="gpt-4o", temperature=0.3, api_key=key)
    else:
        raise ValueError(f"Unknown PROVIDER: {PROVIDER}")


def get_text(message) -> str:
    """claude-sonnet-5 returns content as a list of blocks (thinking + text)
    when extended thinking is on; this extracts just the visible text."""
    content = message.content
    if isinstance(content, str):
        return content
    parts = [block["text"] for block in content if isinstance(block, dict) and block.get("type") == "text"]
    return "\n".join(parts)


llm = get_llm()
print(f"Using provider: {PROVIDER}, model ready: {llm.model if hasattr(llm, 'model') else llm.model_name}")


Using provider: anthropic, model ready: claude-sonnet-5


## Part 1 — Theory: "State" vs. "Memory" (the distinction everything else rests on)

These two words get used interchangeably in casual conversation about
agents, but in LangGraph they mean genuinely different things, and mixing
them up is the single most common source of confusion when people start
adding memory to agents.

| | **Checkpointed state** | **Long-term memory** |
|---|---|---|
| Scope | One `thread_id` (one conversation) | Across *all* threads, tied to a user/entity |
| What's stored | The **entire graph state** — every field in your `State` schema, as of each superstep | **Extracted facts** — a curated, deduplicated summary of what matters |
| Granularity | Full snapshot, replayable step by step | Distilled knowledge, not a transcript |
| Lifetime | Until you delete the thread | Indefinite, meant to outlive any single conversation |
| Mechanism in LangGraph | `checkpointer` argument to `.compile()` (`InMemorySaver`, `SqliteSaver`, ...) | Not built into core LangGraph — you layer a separate memory system (Mem0, a vector store, LangGraph's `Store` interface, etc.) |
| Real-life analogy | Your browser tab staying open with everything still on screen | A person remembering *you* — your name, preferences — even after the conversation ended and a new one started |

```text
+-----------------------------+          +-----------------------------+
|   CHECKPOINTED STATE        |          |   LONG-TERM MEMORY          |
|   (thread-scoped)           |          |   (user-scoped)             |
|                              |          |                              |
|  thread_id="conv-1"         |          |  user_id="alex"              |
|  [msg1, msg2, msg3, ...]    |          |  "prefers email support"     |
|  full replayable snapshot   |          |  "Enterprise plan"           |
|                              |          |  extracted, deduplicated     |
|  Gone/irrelevant the moment |          |  Survives across EVERY       |
|  you start thread_id=       |          |  future thread_id for        |
|  "conv-2"                   |          |  this same user               |
+-----------------------------+          +-----------------------------+
```

**Why this distinction matters in production**: a support chatbot needs
*both*. Within one conversation, it needs full state (checkpointing) so it
doesn't lose track of what's already been said or done. Across
conversations — the customer comes back next week — it needs long-term
memory, because nobody wants to re-explain their subscription tier every
time they open a new chat.

**Easy to confuse with:**
- *Long-term memory is not "a bigger checkpointer."* Making checkpoints
  persist forever doesn't give you cross-thread recall — a checkpoint is
  still keyed by `thread_id`, and a new thread simply doesn't look at it.
- *Long-term memory is not RAG over documents.* RAG retrieves knowledge
  *about the world*; this kind of memory retrieves facts *about the user/
  entity you're talking to*, extracted from your own conversations with
  them.


## Part 2 — Short-term memory: `InMemorySaver`

### Theory

A **checkpointer** is the object LangGraph uses to save the graph's
`State` after every superstep, keyed by `(thread_id, checkpoint_id)`. Pass
one to `.compile(checkpointer=...)` and every `.invoke()`/`.stream()` call
that includes a `thread_id` in its config will:

1. Load the latest checkpoint for that `thread_id` (if any) before running.
2. Run the graph.
3. Save a new checkpoint after every superstep — not just at the end, which
   is exactly what makes `interrupt()`/resume possible: the graph can pause
   mid-execution and the paused state is already durably (or in this case,
   in-memory) saved.

`InMemorySaver` keeps every checkpoint in a plain Python dict inside the
current process. It is the simplest possible checkpointer — zero setup,
perfect for notebooks, tests, and local development.

```text
   thread_id="support-1"
   +------------------------------------------------+
   |  checkpoint 1: {messages: [human: "hi"]}        |
   |  checkpoint 2: {messages: [..., ai: "hello!"]}   |
   |  checkpoint 3: {messages: [..., human: "..."]}   |  <- current
   +------------------------------------------------+
        all living in a Python dict, in this process's RAM
```

**The catch**: restart the Python process (kernel restart, server
redeploy, crash) and the dict is gone. Every conversation, gone. This is
fine for a notebook demo; it is *not* fine for a production chatbot.


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage


class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


def chat_node(state: ChatState) -> ChatState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


builder = StateGraph(ChatState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

memory_saver = InMemorySaver()
graph_inmemory = builder.compile(checkpointer=memory_saver)

print("Graph compiled with InMemorySaver.")


Graph compiled with InMemorySaver.


### Mermaid — graph topology (same simple chat node; the checkpointer wraps *around* it, not visible in the topology itself)

```mermaid
graph LR
    START([START]) --> chat[chat node<br/>calls LLM]
    chat --> END([END])
```

### Real-life example: a support chat that remembers within one conversation

We'll run a 3-turn conversation on `thread_id="support-1"` and confirm the
agent recalls what the customer said two turns ago — all within this one
process. **This cell makes 3 real Claude API calls (one per turn).**


In [ ]:
config_a = {"configurable": {"thread_id": "support-1"}}

turn1 = graph_inmemory.invoke(
    {"messages": [HumanMessage(content="Hi, I'm Priya and I'm on the Enterprise plan.")]},
    config=config_a,
)
print("Turn 1:", get_text(turn1["messages"][-1]), "\n")

turn2 = graph_inmemory.invoke(
    {"messages": [HumanMessage(content="My API requests are timing out, can you help?")]},
    config=config_a,
)
print("Turn 2:", get_text(turn2["messages"][-1]), "\n")

turn3 = graph_inmemory.invoke(
    {"messages": [HumanMessage(content="What's my name and plan again? Just answer in one short sentence.")]},
    config=config_a,
)
print("Turn 3 (recall check):", get_text(turn3["messages"][-1]))


Turn 1: Hi Priya, welcome! Good to have you here on the Enterprise plan.

I should mention, though — I don't actually have access to account systems, so I can't verify plan details or pull up any account-specific information on my end. If you need something that depends on your Enterprise plan specifically (like usage limits, billing details, dedicated support channels, or feature access), you may need to check your account dashboard or reach out to your account team/support for accurate info.

That said, I'm happy to help with whatever brought you here today — whether it's a question, a task, or something you're working on. What's on your mind? 

Turn 2: Yes, let's dig into it. Since I don't have access to your account, logs, or system status, I can't check things from my end — but I can help you troubleshoot methodically. A few questions to narrow it down:

**Basics:**
1. Which API/endpoint(s) are timing out? (all of them, or specific ones?)
2. When did this start? Suddenly, or gradu

**Expected output**: Turn 3's answer correctly states "Priya" and
"Enterprise" — proving the checkpointer replayed the full message history
for `thread_id="support-1"` before this call, even though we never
manually re-sent turns 1 and 2.

### Interrupt + resume, still within `InMemorySaver`

Now we prove the *other* half of the checkpointer's job: pausing execution
mid-graph and resuming exactly where it left off. We build a second, small
graph that deliberately interrupts to ask a human for approval before
"escalating" a ticket.


In [ ]:
from langgraph.types import interrupt, Command


class EscalationState(TypedDict):
    issue: str
    approved: bool
    outcome: str


def draft_escalation(state: EscalationState) -> EscalationState:
    return {"issue": state["issue"]}


def human_approval(state: EscalationState) -> EscalationState:
    decision = interrupt({"question": f"Approve escalation for: {state['issue']}?"})
    return {"approved": decision}


def act_on_decision(state: EscalationState) -> EscalationState:
    if state["approved"]:
        return {"outcome": "Escalated to Tier 2 support."}
    return {"outcome": "Resolved at Tier 1, no escalation needed."}


esc_builder = StateGraph(EscalationState)
esc_builder.add_node("draft", draft_escalation)
esc_builder.add_node("approval", human_approval)
esc_builder.add_node("act", act_on_decision)
esc_builder.add_edge(START, "draft")
esc_builder.add_edge("draft", "approval")
esc_builder.add_edge("approval", "act")
esc_builder.add_edge("act", END)

esc_graph_inmemory = esc_builder.compile(checkpointer=InMemorySaver())

esc_config = {"configurable": {"thread_id": "escalation-1"}}

# First call runs until it hits interrupt() and pauses there.
result = esc_graph_inmemory.invoke({"issue": "Data export API returning 500s for 2 hours"}, config=esc_config)
print("Paused. Interrupt payload:", result["__interrupt__"])


Paused. Interrupt payload: [Interrupt(value={'question': 'Approve escalation for: Data export API returning 500s for 2 hours?'}, id='45bfa8fecbfedca7ec3c461a94fb0086')]


```mermaid
graph LR
    START([START]) --> draft[draft node]
    draft --> approval[approval node<br/>calls interrupt]
    approval -.pauses here,<br/>state checkpointed.-> approval
    approval --> act[act node]
    act --> END([END])
```

The `interrupt()` call inside `approval` is what pauses execution — the
checkpointer is what makes the pause safe to resume from later, on the
same `thread_id`, via `Command(resume=...)`.


The graph is now **paused mid-execution**, with its state durably
held by the checkpointer (durable for this process's lifetime — that
caveat matters, see the next section). We resume it by sending a
`Command(resume=...)` on the *same* `thread_id` — no re-running from
scratch.

In [ ]:
resumed = esc_graph_inmemory.invoke(Command(resume=True), config=esc_config)
print("Resumed outcome:", resumed["outcome"])


Resumed outcome: Escalated to Tier 2 support.


**Expected output**: `"Escalated to Tier 2 support."` — the graph
resumed exactly at the `act` node, using the `approved=True` we injected
via `Command(resume=...)`, without re-running `draft` or re-asking for
approval.

### Common errors
- Forgetting to pass `config` (with `thread_id`) to `.invoke()` — without
  it, LangGraph either errors or treats every call as a brand-new thread,
  and nothing appears to be remembered.
- Reusing the *same* `thread_id` across genuinely unrelated conversations
  — the agent will "remember" things from a different topic and confuse
  itself. `thread_id` should map 1:1 to one real conversation.
- Assuming `InMemorySaver` state survives a notebook kernel restart. It
  will not — that's exactly what the next section demonstrates.


## Part 3 — Short-term memory: `SqliteSaver` (durable across restarts)

### Theory

`SqliteSaver` (from `langgraph-checkpoint-sqlite`) implements the exact
same checkpointer interface as `InMemorySaver`, but writes every
checkpoint to a real SQLite file on disk instead of an in-process dict.
Same API, same `thread_id`-keyed model — the only thing that changes is
*where* the bytes live.

```text
   thread_id="support-1"                    agent_memory.sqlite (on disk)
   +----------------------+     writes      +---------------------------+
   |  checkpoint 1         | --------------> |  checkpoints table        |
   |  checkpoint 2         |                 |  writes table              |
   |  checkpoint 3         |                 |  ... survives process exit |
   +----------------------+                 +---------------------------+
```

This is the difference that matters for production: if the Python process
restarts (crash, redeploy, scheduled restart), a `SqliteSaver` pointed at
the same file picks up exactly where it left off — including mid-interrupt
state. We'll prove this for real, not just claim it: we'll interrupt a
graph, then **throw away the Python object entirely and rebuild it from
scratch**, pointed at the same file, and resume.


In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

DB_PATH = "agent_memory_deepdive.sqlite"

# Fresh file each run of this notebook, so the demo is reproducible.
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
sqlite_saver = SqliteSaver(conn)

esc_graph_sqlite = esc_builder.compile(checkpointer=sqlite_saver)

sqlite_config = {"configurable": {"thread_id": "escalation-durable-1"}}

paused = esc_graph_sqlite.invoke(
    {"issue": "Customer billed twice for the same invoice"}, config=sqlite_config
)
print("Paused (durable). Interrupt payload:", paused["__interrupt__"])

conn.close()
print(f"Connection closed. {DB_PATH} now holds the paused checkpoint on disk.")


Paused (durable). Interrupt payload: [Interrupt(value={'question': 'Approve escalation for: Customer billed twice for the same invoice?'}, id='724724f20a8dc5db4ba47367848860dc')]
Connection closed. agent_memory_deepdive.sqlite now holds the paused checkpoint on disk.


Now we simulate a real restart: **new SQLite connection, new
`SqliteSaver`, new compiled graph object** — nothing from the Python
objects above is reused, only the file path.

In [ ]:
# --- simulating a fresh process: nothing above this line is reused except the file path ---
conn2 = sqlite3.connect(DB_PATH, check_same_thread=False)
sqlite_saver_2 = SqliteSaver(conn2)
esc_graph_sqlite_restarted = esc_builder.compile(checkpointer=sqlite_saver_2)

resumed = esc_graph_sqlite_restarted.invoke(Command(resume=True), config=sqlite_config)
print("Resumed after 'restart':", resumed["outcome"])

conn2.close()


Resumed after 'restart': Escalated to Tier 2 support.


**Expected output**: `"Escalated to Tier 2 support."` — resumed
correctly even though every Python object was rebuilt from scratch, proving
persistence is on disk, not in the process.

### `InMemorySaver` vs. `SqliteSaver` — when to use which

| | `InMemorySaver` | `SqliteSaver` |
|---|---|---|
| Durability | None — gone on process exit | Survives process restart (single file) |
| Setup cost | Zero | Minimal — one file path |
| Speed | Fastest (RAM) | Fast, small disk I/O overhead |
| Multi-process / multi-worker safe? | No — each process has its own dict | Partially — SQLite file locking allows limited concurrent access, but it's not built for high-concurrency multi-instance deployments |
| Best for | Notebooks, unit tests, local dev, short-lived scripts | Single-instance production services, local desktop apps, demos that must survive a restart |
| Not sufficient for | Anything needing durability | Multi-instance/horizontally-scaled deployments (many API server replicas) — needs a shared backend like `PostgresSaver` or a Redis-backed checkpointer instead |

**Rule of thumb**: start with `InMemorySaver` while developing. Switch to
`SqliteSaver` the moment you need a demo, a local tool, or a single-process
service to survive a restart. Graduate to a `PostgresSaver` (or similar
shared-database checkpointer, not built in this notebook) only once you're
running more than one instance of your service and they all need to see
the same threads.

### Common errors
- Opening a new `sqlite3.connection` per call instead of reusing/reopening
  against the *same file* — checkpoints are only durable if every restart
  points at the same `DB_PATH`.
- `check_same_thread=False` is required if the connection is touched from
  more than one thread (e.g. inside a web framework's request handling) —
  omitting it raises an error the first time that happens.
- Forgetting SQLite is still a *single-file, single-machine* store — it
  does not make your app horizontally scalable.


## Part 4 — The amnesia problem: why checkpointing alone isn't enough

Both checkpointers above are keyed by `thread_id`. Start a *new*
`thread_id` for the *same real customer*, and the agent has no idea who
they are — no matter how durable the checkpointer is.


In [ ]:
# Same customer, Priya, but a brand-new conversation thread — e.g. she opened
# a new browser tab, or the old ticket was closed and she opened a new one.
new_thread_config = {"configurable": {"thread_id": "support-2-brand-new"}}

amnesia_check = graph_inmemory.invoke(
    {"messages": [HumanMessage(content="What's my name and plan? Just answer in one short sentence.")]},
    config=new_thread_config,
)
print("New thread response:", get_text(amnesia_check["messages"][-1]))


New thread response: I don't have any information about your name or plan since this is the start of our conversation—could you share those details?


**Expected output**: the agent admits it doesn't know — it has no
information about Priya in this thread, even though `thread_id="support-1"`
(same process, same `InMemorySaver` instance!) has her full conversation
sitting right there. Durability was never the problem here; **scope** is.

This is exactly the gap long-term memory fills.


## Part 5 — Long-term memory: Mem0 + ChromaDB

### Theory

**Mem0** is a memory-management library: you feed it conversation turns,
it uses an LLM to extract durable facts from them, deduplicates/updates
against what it already knows about that `user_id`, and stores the result
as embeddings in a vector store. You then query it — semantically, in
plain English — for anything relevant to the current turn, and inject the
result into your prompt.

**ChromaDB** is the vector store Mem0 uses to actually persist and search
those embeddings on disk. Mem0 handles *what* to remember and *how to
update it*; Chroma handles *where the vectors physically live and how
they're searched*.

```text
 conversation turn                      querying later (any thread)
 ------------------                      ---------------------------
 "I'm on Enterprise,                     "What plan is this user on?"
  prefer email support"                            |
        |                                            v
        v                                   +-----------------+
  +------------+   extracts facts           |  Mem0 .search()  |
  |  Mem0.add()| ------------------->        |  embeds query,    |
  |  (LLM call)|                             |  searches Chroma   |
  +------------+                             +--------+---------+
        |                                             |
        v                                             v
  +------------------------+          "User is on the Enterprise plan"
  |  ChromaDB (on disk)     |  <---   relevant facts returned, ranked
  |  embedded fact vectors, |         by similarity
  |  keyed by user_id       |
  +------------------------+
```

**Where this plugs into a LangGraph agent**: as a step *before* the LLM
call (retrieve relevant facts, inject into the system/context) and a step
*after* it (extract and store any new facts from this turn) — conceptually
identical to a RAG retrieval step, except retrieving *about the user*
instead of *about documents*.

### Config note (from research against current Mem0 docs)

Mem0's `Memory()` defaults to OpenAI for both the LLM and the embedder,
and Qdrant as the vector store. To keep this notebook fully within the
`PROVIDER="anthropic"` path and avoid a second paid API purely for
embeddings, we override all three explicitly:


In [ ]:
from mem0 import Memory

MEM0_CHROMA_PATH = "./mem0_chroma_store"

# Mem0 is wired to the same PROVIDER flag as the chat LLM above, so switching
# PROVIDER at the top of the notebook switches everything consistently — no
# mixed-provider surprises.
if PROVIDER == "anthropic":
    mem0_config = {
        "llm": {
            "provider": "anthropic",
            "config": {"model": "claude-sonnet-5", "temperature": 0.1},
        },
        "embedder": {
            # Local, free, no API key — avoids depending on OpenAI just for
            # embeddings when the chat path is Anthropic-only.
            "provider": "huggingface",
            "config": {"model": "sentence-transformers/all-MiniLM-L6-v2"},
        },
        "vector_store": {
            "provider": "chroma",
            "config": {"collection_name": "agent_memory_deepdive", "path": MEM0_CHROMA_PATH},
        },
    }
    embedder_note = "huggingface (local)"
elif PROVIDER == "openai":
    mem0_config = {
        "llm": {
            "provider": "openai",
            "config": {"model": "gpt-4o", "temperature": 0.1},
        },
        "embedder": {
            "provider": "openai",
            "config": {"model": "text-embedding-3-small"},
        },
        "vector_store": {
            "provider": "chroma",
            "config": {"collection_name": "agent_memory_deepdive", "path": MEM0_CHROMA_PATH},
        },
    }
    embedder_note = "openai (text-embedding-3-small)"
else:
    raise ValueError(f"Unknown PROVIDER: {PROVIDER}")

memory = Memory.from_config(mem0_config)
print(f"Mem0 initialized: llm={PROVIDER}, embedder={embedder_note}, vector_store=chroma (local disk).")


/Users/utsabchakraborty/Documents/Edureka_Full_Course/Live_Class_Codes/Coding_Agent_Enabled_Demo/teaching/langgraph_basics/.venv/lib/python3.14/site-packages/chromadb/telemetry/opentelemetry/__init__.py:128: DeprecationWarning: 'asyncio.iscoroutinefunction' is deprecated and slated for removal in Python 3.16; use inspect.iscoroutinefunction() instead
  if asyncio.iscoroutinefunction(f):


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7951.21it/s]

/Users/utsabchakraborty/Documents/Edureka_Full_Course/Live_Class_Codes/Coding_Agent_Enabled_Demo/teaching/langgraph_basics/.venv/lib/python3.14/site-packages/mem0/embeddings/huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimension()
The 'chroma' vector store does not support keyword search. Hybrid (BM25) scoring will be disabled and search will use semantic similarity only. To enable hybrid search, switch to a store with keyword_search support (e.g. qdrant, elasticsearch, pgvector).


Mem0 initialized: llm=anthropic/claude-sonnet-5, embedder=huggingface (local), vector_store=chroma (local disk).


### Real-life example: profile memory across two *different* conversation threads

Turn 1 happens in `thread_id="support-1"` (this week). We store what we
learn about Priya into Mem0, keyed by `user_id="priya"` — not by
`thread_id`. **This cell makes 1 real Claude API call** (Mem0's
fact-extraction step inside `.add()`).


In [ ]:
add_result = memory.add(
    [
        {"role": "user", "content": "Hi, I'm Priya, I'm on the Enterprise plan, and I prefer email over phone support."},
        {"role": "assistant", "content": "Thanks Priya, noted — Enterprise plan, email preferred. How can I help today?"},
    ],
    user_id="priya",
)
print("Facts extracted and stored:")
for r in add_result["results"]:
    print(" -", r["memory"])


Failed to load spaCy lemma model: spaCy is not installed. Install it with: pip install mem0ai[nlp]


Failed to load spaCy full model: spaCy is not installed. Install it with: pip install mem0ai[nlp]


Facts extracted and stored:
 - User's name is Priya
 - Priya is on the Enterprise plan
 - Priya prefers email over phone support


Weeks later, Priya opens a **brand-new thread** — `thread_id=
"support-99-new-conversation"`. A plain checkpointer would know nothing
here (exactly like Part 4). We now build a small LangGraph agent that,
before calling the LLM, retrieves relevant Mem0 facts for `user_id="priya"`
and injects them into the system prompt. **This cell makes 1 real Claude
API call** (the chat turn itself; the `memory.search()` retrieval step is
a local embedding search, no LLM call).


In [ ]:
from langchain_core.messages import SystemMessage


class ProfileAwareState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str


def retrieve_and_respond(state: ProfileAwareState) -> ProfileAwareState:
    last_user_msg = state["messages"][-1].content
    relevant = memory.search(last_user_msg, filters={"user_id": state["user_id"]}, limit=5)
    facts = [r["memory"] for r in relevant["results"]]

    system_context = (
        "Known facts about this user from prior conversations: "
        + ("; ".join(facts) if facts else "none yet")
    )
    response = llm.invoke([SystemMessage(content=system_context)] + state["messages"])
    return {"messages": [response]}


profile_builder = StateGraph(ProfileAwareState)
profile_builder.add_node("respond", retrieve_and_respond)
profile_builder.add_edge(START, "respond")
profile_builder.add_edge("respond", END)

# Note: no checkpointer needed to prove cross-thread recall here — the point
# is that recall comes from Mem0 (user_id-scoped), not from graph state.
profile_graph = profile_builder.compile()

new_thread_result = profile_graph.invoke(
    {
        "messages": [HumanMessage(content="What plan am I on, and how do I prefer to be contacted? One short sentence.")],
        "user_id": "priya",
    }
)
print("Brand-new thread, weeks later:", get_text(new_thread_result["messages"][-1]))


Brand-new thread, weeks later: You're on the Enterprise plan, and you prefer to be contacted via email.


```mermaid
graph LR
    START([START]) --> respond[respond node<br/>1. Mem0.search facts for user_id<br/>2. inject facts as SystemMessage<br/>3. LLM call]
    respond --> END([END])
    Mem0Store[(Mem0 + ChromaDB<br/>user_id-scoped)] -.search.-> respond
```

Notice this graph has **no checkpointer at all** — that's deliberate. The
recall you're about to see comes entirely from Mem0/Chroma keyed on
`user_id`, not from any thread-scoped state, which is exactly the point
this section is proving.


**Expected output**: the agent correctly answers "Enterprise plan"
and "email" — recalled entirely from Mem0/Chroma, in a thread that has
*zero* checkpointed history. This is the capability checkpointing alone
can never provide.

### Common errors
- Forgetting `user_id` on either `.add()` or `.search()` — Mem0 partitions
  memory by `user_id`; omit it and facts either aren't scoped correctly or
  aren't found.
- Treating Mem0 as a full transcript store — it deliberately extracts and
  condenses facts. Don't expect verbatim message recall from it; that's
  what checkpointing is for.
- Not overriding the default embedder — leaving Mem0 on its OpenAI default
  silently requires an `OPENAI_API_KEY` even if your chat LLM is Anthropic.


## Part 6 — Alternatives to Mem0, compared

Mem0 is one opinionated way to do long-term memory. It's worth knowing
what else exists and when each is the better fit — this section is theory
only, no additional API cost.

| Approach | What it is | Strengths | Trade-offs | Best when |
|---|---|---|---|---|
| **Mem0** (used above) | Managed extraction + dedup + update logic on top of a vector store | Fast to integrate, handles fact extraction/conflict resolution/consolidation for you | Extra dependency, opinionated extraction you don't fully control | You want cross-thread user memory working quickly, without writing your own extraction/dedup logic |
| **LangGraph's native `Store` interface** (`InMemoryStore` / `PostgresStore`) | LangGraph's own long-term-memory primitive — you write embeddings in and query them out yourself | No extra framework, stays inside LangGraph's own abstractions, full control over what gets stored | You write the extraction prompt, dedup, and update logic yourself — nothing is automatic | You're already deep in LangGraph, want zero extra dependencies, and are fine owning the extraction logic |
| **Zep** | Purpose-built conversational memory service with temporal knowledge graphs | Strong at reasoning over *how facts changed over time* ("what did they used to prefer vs. now"), not just current facts | Another service to run/manage, heavier than Mem0 for simple fact recall | Temporal/relationship reasoning over memory actually matters to the product, not just "remember the latest fact" |
| **Raw vector DB + hand-rolled extraction prompt** (Chroma/Pinecone/Weaviate directly) | You write your own "extract facts from this conversation" prompt and store/query the vectors yourself, no memory-management library at all | Maximum control, zero framework lock-in, extraction logic can be arbitrarily custom | You own everything: extraction, conflict resolution, staleness, dedup — all from scratch | Your extraction rules are unusual enough that a generic library (Mem0 or otherwise) fights you rather than helping |

### Recommendation

- **Default choice for most projects**: Mem0. It gets you working
  cross-thread memory fastest, and its extraction/dedup logic is good
  enough for the common case (a support agent, a personal assistant, a
  companion app remembering user facts).
- **Already using LangGraph and want one fewer dependency**: LangGraph's
  own `Store` interface — same underlying idea, no external library, but
  you write the extraction prompt yourself.
- **Temporal reasoning is a product requirement** (e.g. "the user's
  preference changed last month, and that change itself matters"): Zep.
- **Your extraction needs are genuinely unusual** (e.g. domain-specific
  structured facts a generic prompt won't capture well): raw vector DB +
  a custom-written extraction prompt, accepting you own the whole pipeline.


## Part 7 — Combined finale: checkpointer + Mem0 working together

The realistic pattern: **checkpointing handles within-conversation state,
Mem0+Chroma handles across-conversation recall — used together, not as
alternatives to each other.**

Scenario: Priya opens a support conversation, mentions a new detail
(`billing contact changed to David`). Within that same thread, the agent
should recall it turn-to-turn (checkpointer). In a follow-up conversation
opened days later, the agent should still recall it (Mem0) — proving both
systems are doing their job simultaneously. **This cell makes up to 3 real
Claude API calls** (2 chat turns in-thread + 1 in the new thread; the
Mem0 `.add()` fact-extraction call is included in one of those turns via
the node below).


In [ ]:
class CombinedState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str


def combined_node(state: CombinedState) -> CombinedState:
    last_user_msg = state["messages"][-1].content

    # Long-term recall (Mem0/Chroma) — cross-thread facts about this user.
    relevant = memory.search(last_user_msg, filters={"user_id": state["user_id"]}, limit=5)
    facts = [r["memory"] for r in relevant["results"]]
    system_context = "Known facts about this user: " + ("; ".join(facts) if facts else "none yet")

    # Short-term context (checkpointer) — the rest of state["messages"] already
    # contains this thread's full history, restored automatically by the checkpointer.
    response = llm.invoke([SystemMessage(content=system_context)] + state["messages"])
    response_text = get_text(response)

    # Write new facts back to long-term memory for future threads.
    memory.add(
        [
            {"role": "user", "content": last_user_msg},
            {"role": "assistant", "content": response_text},
        ],
        user_id=state["user_id"],
    )
    return {"messages": [response]}


combined_builder = StateGraph(CombinedState)
combined_builder.add_node("combined", combined_node)
combined_builder.add_edge(START, "combined")
combined_builder.add_edge("combined", END)

combined_graph = combined_builder.compile(checkpointer=InMemorySaver())

# --- Conversation 1, thread "support-week1" ---
week1_config = {"configurable": {"thread_id": "support-week1"}}

r1 = combined_graph.invoke(
    {"messages": [HumanMessage(content="Hi, it's Priya again. My billing contact changed to David.")], "user_id": "priya"},
    config=week1_config,
)
print("Week1 turn1:", get_text(r1["messages"][-1]), "\n")

r2 = combined_graph.invoke(
    {"messages": [HumanMessage(content="Just to confirm — who's the billing contact I just gave you?")], "user_id": "priya"},
    config=week1_config,
)
print("Week1 turn2 (checkpointer recall, same thread):", get_text(r2["messages"][-1]), "\n")


Week1 turn1: Hi Priya, thanks for letting me know! I'd be happy to update your billing contact to David.

To make this change, could you provide a few details:

1. **David's full name** (first and last)
2. **David's email address** (for billing notifications and correspondence)
3. **David's phone number** (optional, but sometimes required for Enterprise billing verification)

Also, just to confirm — should David **replace** you as the sole billing contact, or would you like him **added** as an additional contact while you remain on the account as well?

Once I have these details, I'll get the update processed. And since I know you prefer email, I'll send a confirmation to your email once the change is complete. 



Week1 turn2 (checkpointer recall, same thread): You just told me your billing contact changed to **David** — but I don't have his last name, email, or phone number yet.

If you can share those details, I'll get the update finalized and send a confirmation to your email. 



```mermaid
graph LR
    START([START]) --> combined[combined node<br/>1. Mem0.search - cross-thread facts<br/>2. LLM call - uses facts + thread history<br/>3. Mem0.add - writes new facts back]
    combined --> END([END])
    ThreadCP[(Checkpointer<br/>thread-scoped)] -.restores/saves<br/>message history.-> combined
    Mem0Store[(Mem0 + ChromaDB<br/>user_id-scoped)] -.search / add.-> combined
```

Both memory systems attach to the *same* node here — the checkpointer
handles `state["messages"]` transparently (via `compile(checkpointer=...)`),
while Mem0/Chroma is called explicitly inside the node body. This is the
realistic shape: one graph, two memory systems, each doing the part the
other can't.


Now a **new thread, days later** — proving Mem0 recall works even
though this thread's checkpointed history is completely empty.

In [ ]:
week2_config = {"configurable": {"thread_id": "support-week2-new-thread"}}

r3 = combined_graph.invoke(
    {"messages": [HumanMessage(content="Quick check before I update our records — who's our billing contact on file?")], "user_id": "priya"},
    config=week2_config,
)
print("Week2, brand-new thread (Mem0 recall):", get_text(r3["messages"][-1]))


Week2, brand-new thread (Mem0 recall): David is the billing contact on file for your account.

Let me know if you need anything else updated!


**Expected output**: turn 2 (same thread) recalls "David" via the
checkpointer's replayed message history; the week-2 answer (new,
checkpoint-empty thread) *also* recalls "David" — but this time purely via
Mem0/Chroma's cross-thread fact store. Two different mechanisms, same
correct answer, each doing the job the other one can't.

## Revision summary

- **Checkpointer** (`InMemorySaver`, `SqliteSaver`, ...) = full state,
  scoped to one `thread_id`, the mechanism behind multi-turn recall and
  interrupt/resume.
- **`InMemorySaver`**: zero setup, gone on restart — dev/test/notebooks.
- **`SqliteSaver`**: file-backed, survives restart — single-instance
  production/local tools. Neither scales to multiple concurrent service
  instances; that needs a shared-database checkpointer (e.g. Postgres).
- **Long-term memory** (Mem0 + Chroma here) = extracted facts, scoped to a
  `user_id`, persisting across every future thread — solves the "new
  thread = total amnesia" problem checkpointing cannot solve by itself.
- **Mem0** is one option among several (LangGraph `Store`, Zep, raw vector
  DB + custom prompt) — pick based on how much control vs. convenience you
  need, and whether temporal reasoning over memory matters.
- In a real system, both layers run together: checkpointer for
  within-conversation continuity, long-term memory for across-conversation
  recall.

## Explain like I'm 12

Imagine you're talking to a friend on the phone. While you're on the call,
they remember everything you've said *this call* — that's the
checkpointer. But your friend also just *knows* things about you from
before — your birthday, your favorite food — even on a call you've never
had this exact conversation on before. That "just knowing you" part is
long-term memory. A checkpointer is great at "remember this conversation."
Long-term memory is what lets someone remember *you*.

## Explain for interview

"LangGraph checkpointers persist full graph state keyed by `thread_id`,
enabling multi-turn continuity and interrupt/resume within a single
conversation — but they're conversation-scoped by design, not
user-scoped. Production agents typically need a second layer: a long-term
memory system (Mem0, LangGraph's `Store`, or a custom vector-store +
extraction pipeline) that distills and persists facts about an entity
(usually the user) keyed independently of any one thread, so an agent
recognizes returning users across brand-new conversations. I'd choose
`InMemorySaver` for development, a durable single-file/single-instance
checkpointer like `SqliteSaver` for a single-process production service,
and a shared-database checkpointer (Postgres-backed) once the service runs
multiple instances — and I'd add Mem0 or an equivalent specifically to
close the cross-thread recall gap that no checkpointer, however durable,
can close by itself."

## Glossary

- **Checkpointer** — LangGraph component that persists graph state per
  `thread_id` after each superstep.
- **`thread_id`** — the key that scopes checkpointed state to one
  conversation.
- **`user_id`** — the key Mem0 uses to scope long-term facts to one
  entity, independent of any thread.
- **Superstep** — one round of node execution in LangGraph's execution
  model; checkpoints are written after each one.
- **`interrupt()`** — pauses graph execution at a node, persisting state
  via the checkpointer so it can be resumed later with `Command(resume=...)`.
- **Fact extraction** — the LLM-driven step (inside Mem0's `.add()`) that
  turns raw conversation text into concise, storable facts.
- **Vector store** — a database (Chroma here) that stores embeddings and
  supports semantic similarity search; the storage layer under Mem0.
- **Embedder** — the model that converts text into a vector for storage/
  search; overridden to a local HuggingFace model here to avoid a second
  paid API.

## Checkpoint questions

1. **Q: What's the key difference between checkpointed state and
   long-term memory?**
   A: Checkpointed state is a full, thread-scoped snapshot of everything
   in the graph's state; long-term memory is a distilled, user-scoped set
   of facts meant to persist across many threads.

2. **Q: Why does a new `thread_id` cause an agent to "forget" a customer
   even with `SqliteSaver` in use?**
   A: Because checkpoints are looked up by `thread_id` — a new thread has
   no checkpoint history to load, regardless of how durable the storage
   backend is.

3. **Q: When would you choose `SqliteSaver` over `InMemorySaver`?**
   A: Any time the process might restart and you need the conversation
   (including mid-interrupt state) to survive that restart — e.g. a
   single-instance production service or a local desktop tool.

4. **Q: Why doesn't `SqliteSaver` scale to multiple service instances?**
   A: It's a single local file; concurrent writers across separate
   machines/processes aren't what it's built for. Multi-instance
   deployments need a shared-database checkpointer such as one backed by
   Postgres.

5. **Q: What does `interrupt()` actually rely on to make resume possible?**
   A: The checkpointer — execution pauses at the interrupt point, and the
   state at that point is already saved, so `Command(resume=...)` can
   continue from exactly there.

6. **Q: Why did we override Mem0's default embedder in this notebook?**
   A: Mem0 defaults to an OpenAI embedder; overriding it to a local
   HuggingFace model keeps the whole notebook on a single provider
   (Anthropic for chat) without silently requiring a second paid API key.

7. **Q: What does `memory.add()` actually do internally?**
   A: It calls an LLM to extract durable facts from the given
   conversation turns, reconciles them against existing facts for that
   `user_id` (updating/deduplicating), and stores the result as embeddings
   in the configured vector store.

8. **Q: In the combined-finale example, which mechanism made turn 2 (same
   thread) work, and which made the week-2 answer (new thread) work?**
   A: Turn 2 relied on the checkpointer replaying that thread's message
   history; the week-2 answer relied on Mem0/Chroma's `user_id`-scoped
   fact search, since that thread had no checkpointed history at all.

9. **Q: Name one alternative to Mem0 and when you'd pick it instead.**
   A: LangGraph's native `Store` interface — pick it when you want to stay
   entirely within LangGraph's own abstractions and are willing to write
   your own fact-extraction/dedup logic instead of relying on a library
   to do it for you. (Zep is another valid answer, for temporal reasoning
   needs.)

10. **Q: Is long-term memory (as built here) a replacement for
    checkpointing, or a complement to it?**
    A: A complement — production agents need both: checkpointing for
    reliable within-conversation continuity (including interrupt/resume),
    and long-term memory for recognizing the same user across brand-new
    conversations.
